In [1]:
import xarray as xr

In [2]:
data = xr.open_zarr("/scratch/as15415/Data/Emulation_Data/OM4_Horizontal_Regrid_Old.zarr")

### Convert and save surface data to /vast

Make sure you are using atleast 8 cores!

In [4]:
data['so'] = data['so'][:,0]
data['thetao'] = data['thetao'][:,0]
data['uo'] = data['uo'][:,0]
data['vo'] = data['vo'][:,0]

In [5]:
from dask.diagnostics import ProgressBar

In [6]:
with ProgressBar():
    data_mean = data.mean().compute()

[########################################] | 100% Completed | 259.26 s


In [7]:
data_mean.to_zarr('/vast/sd5313/data/m2lines/3D_ocean_data/surface_data_means')

In [8]:
with ProgressBar():
    data_std = data.std().compute()

[########################################] | 100% Completed | 244.16 s


In [9]:
data_std.to_zarr('/vast/sd5313/data/m2lines/3D_ocean_data/surface_data_stds')

In [10]:
data.to_zarr('/vast/sd5313/data/m2lines/3D_ocean_data/surface_data')

### Tests

In [3]:
data

<xarray.Dataset>
Dimensions:  (time: 4745, y: 180, x: 360, lev: 19)
Coordinates:
  * time     (time) object 1958-01-03 12:00:00 ... 2022-12-29 12:00:00
  * x        (x) float64 0.5 1.5 2.5 3.5 4.5 ... 355.5 356.5 357.5 358.5 359.5
  * y        (y) float64 -89.5 -88.5 -87.5 -86.5 -85.5 ... 86.5 87.5 88.5 89.5
Dimensions without coordinates: lev
Data variables:
    hfds     (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so       (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    tauuo    (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    tauvo    (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    thetao   (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    uo       (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    vo       (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    zos      (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>

In [4]:
data.so["lev"]

<xarray.DataArray 'lev' (lev: 19)>
array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18])
Dimensions without coordinates: lev

In [14]:
def test_gen(input_vars, extra_vars, output_vars, lag=1, depth_mode="all"):
    data = xr.open_zarr(
        "/scratch/as15415/Data/Emulation_Data/OM4_Horizontal_Regrid_Old.zarr"
    )

    inputs = []
    extra_in = []
    outputs = []

    for var in input_vars:
        if var == "zos":
            inputs.append(data[var])
        elif depth_mode == "surface":
            inputs.append(data[var][:, 0])
        elif depth_mode == "all":
            for i in range(data[var].shape[1]):
                inputs.append(data[var][:, i])

    for var in extra_vars:
        extra_in.append(data[var])

    for var in output_vars:
        if var == "zos":
            outputs.append(data[var][lag:])
        elif depth_mode == "surface":
            outputs.append(data[var][lag:, 0])
        elif depth_mode == "all":
            for i in range(data[var].shape[1]):
                outputs.append(data[var][lag:, i])

    inputs = tuple(inputs)
    extra_in = tuple(extra_in)
    outputs = tuple(outputs)

    return inputs, extra_in, outputs

In [17]:
inputs, extra_in, outputs = test_gen(
    ["uo", "vo", "thetao", "so", "zos"],
    ["tauuo", "tauvo", "hfds"],
    ["uo", "vo", "thetao", "so", "zos"],
    depth_mode="all",
)

In [20]:
len(inputs)

77

In [8]:
import numpy as np

In [9]:
data.tauuo[:, :, :].values

array([[[        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        ...,
        [ 0.03739167,  0.03725152,  0.03707196, ...,  0.03757205,
          0.03754889,  0.03749075],
        [ 0.02840508,  0.02819717,  0.02796913, ...,  0.02895568,
          0.02878909,  0.02860614],
        [ 0.01770086,  0.01757012,  0.01743707, ...,  0.01812409,
          0.01798942,  0.01784848]],

       [[        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        ...,
        [-0.11497339, -0.11504205, -0.1151281 , ..., -

In [10]:
np.array(data.zos)

array([[[        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        ...,
        [-0.32888971, -0.32703181, -0.32515863, ..., -0.33432069,
         -0.33258432, -0.33075901],
        [-0.29781562, -0.29685536, -0.29586226, ..., -0.30082796,
         -0.29984848, -0.29884841],
        [-0.2679397 , -0.26772973, -0.26750271, ..., -0.2687237 ,
         -0.2684744 , -0.26819819]],

       [[        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        ...,
        [-0.59817378, -0.59610624, -0.59397595, ..., -

In [11]:
data.uo[0].values

array([[[        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        ...,
        [-0.01251889, -0.01230785, -0.01209403, ..., -0.01272803,
         -0.01274474, -0.01267345],
        [-0.01845972, -0.01802384, -0.0176047 , ..., -0.01959374,
         -0.01925796, -0.01886955],
        [-0.02259186, -0.02249625, -0.02240956, ..., -0.02279709,
         -0.02275002, -0.02269251]],

       [[        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        [        nan,         nan,         nan, ...,         nan,
                 nan,         nan],
        ...,
        [-0.0302047 , -0.03006262, -0.02988526, ..., -